### Computing RDF2VEC for WikiData dataset

In [1]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id 
			FROM <wikidata>
			WHERE { 
			{ ?id [] [] }
			UNION
			{ [] ?id [] }
			UNION
			{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
entities = [row["id"] for row in reader]

In [2]:
from pyrdf2vec import RDF2VecTransformer
from pyrdf2vec.embedders import Word2Vec
from pyrdf2vec.graphs import KG
from pyrdf2vec.walkers import RandomWalker

In [3]:
# %%prun -s cumulative

# Define our knowledge graph (here: DBPedia SPARQL endpoint).
knowledge_graph = KG(
    location="http://127.0.0.1:8890/sparql/",
    skip_verify=True,
)

# Create our transformer, setting the embedding & walking strategy.
transformer = RDF2VecTransformer(
    embedder=Word2Vec(epochs=10),
    walkers=[RandomWalker(
        max_depth=4,
        max_walks=10,
        with_reverse=False,
        n_jobs=10,
    )],
    verbose=1
)

# Get our embeddings.
embeddings, literals = transformer.fit_transform(knowledge_graph, entities)

100%|██████████| 934369/934369 [2:03:06<00:00, 126.50it/s] 


Extracted 9337966 walks for 934369 entities (7386.8212s)
Fitted 9337966 walks (458.4700s)


In [ ]:
# Backing up the results into pickle file
import pickle

with open("../datasets/wikidata/rdf2vec100dim.pkl", "wb") as f:
    pickle.dump(
        dict(zip(entities, embeddings)),
        f
    )

### Computing occurrences for WikiData dataset

In [9]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id, COUNT(?id) AS ?count
			FROM <wikidata>
			WHERE { 
				{ ?id [] [] }
				UNION
				{ [] ?id [] }
				UNION
				{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
count_vals = {row["id"]: int(row["count"]) for row in reader}

In [ ]:
import pickle

with open("../datasets/wikidata/counts.pkl", "wb") as f:
	pickle.dump(count_vals, f)